# QUANTAXIS 因子研究 & 策略回测 — 零改造全流程

**数据流**: MongoDB 日线 → 技术指标 → 因子计算 → IC 验证 → 策略回测  
**零依赖**: 不需要 ClickHouse，不需要修改 QA 源码，不需要 QARS2

In [ ]:
import QUANTAXIS as QA
import pandas as pd
import numpy as np
from scipy.stats import spearmanr, pearsonr
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']

print(f'QUANTAXIS: {QA.__version__}')
print(f'QARS 加速: {QA.__has_qars__}')

---
## Part 1: 数据加载

从 MongoDB 获取日线数据。`QA_fetch_stock_day_adv` 返回 `DataStruct` 对象，
支持 `.open` / `.close` / `.high` / `.low` / `.volume` 按列访问。

In [ ]:
# 1.1 获取股票列表
from QUANTAXIS.QAFetch.QAQuery import QA_fetch_stock_list
from QUANTAXIS.QAFetch.QAQuery_Advance import QA_fetch_stock_day_adv

all_stocks = QA_fetch_stock_list()
print(f'全市场股票数: {len(all_stocks)}')
print(all_stocks.head(5))

In [ ]:
# 1.2 选择测试标的和日期范围
# 先用沪深300中有代表性的几只做快速测试
TEST_CODES = ['000001', '000002', '000858', '600036', '600519', '601318']
START_DATE = '2020-01-01'
END_DATE   = '2025-12-31'

print(f'测试标的: {TEST_CODES}')
print(f'时间范围: {START_DATE} ~ {END_DATE}')

In [ ]:
# 1.3 加载日线 DataStruct（带指标计算）
# add_func 链式调用，一次算出所有需要的指标

all_data = {}
for code in TEST_CODES:
    try:
        ds = QA_fetch_stock_day_adv(code, START_DATE, END_DATE)
        ds = ds.add_func(QA.QA_indicator_MA, 5, 10, 20, 60)
        ds = ds.add_func(QA.QA_indicator_MACD)
        ds = ds.add_func(QA.QA_indicator_RSI, 14)
        ds = ds.add_func(QA.QA_indicator_BOLL, 20, 2)
        ds = ds.add_func(QA.QA_indicator_ATR, 14)
        ds = ds.add_func(QA.QA_indicator_KDJ)
        all_data[code] = ds
        print(f'[OK] {code}: {len(ds.data)} 条日线, {ds.data.shape[1]} 列')
    except Exception as e:
        print(f'[FAIL] {code}: {e}')

print(f'\n成功加载 {len(all_data)}/{len(TEST_CODES)} 只股票')

In [ ]:
# 1.4 查看某只股票的数据结构
code = TEST_CODES[0]
ds = all_data[code]
print(f'列名: {ds.data.columns.tolist()}')
print(f'行数: {len(ds.data)}')
print(f'索引: {ds.data.index.names}')
ds.data.tail(5)

---
## Part 2: 因子计算

基于已有指标列，用 pandas 向量化计算三类因子：**动量、趋势、波动**

In [ ]:
def build_factors(ds, code):
    """
    输入: DataStruct（已 add_func 指标）
    输出: DataFrame，包含所有因子值
    """
    df = ds.data.copy()
    close = df['close']
    
    factors = pd.DataFrame(index=df.index)
    factors['code'] = code
    factors['close'] = close
    
    # ============ 动量因子 ============
    factors['momentum_5d']  = close.pct_change(5)                    # 5日收益率
    factors['momentum_20d'] = close.pct_change(20)                   # 20日收益率
    factors['momentum_60d'] = close.pct_change(60)                   # 60日收益率
    factors['rsi_14']       = df.get('RSI14', np.nan)                # 14日RSI (QA内置)
    
    # ============ 趋势因子 ============
    if 'MA20' in df.columns:
        factors['ma_deviation_20'] = (close - df['MA20']) / df['MA20']      # 均线偏离
        factors['ma_deviation_60'] = (close - df['MA60']) / df['MA60']
        factors['ma_cross']        = (df['MA5'] > df['MA20']).astype(int)   # 金叉信号
    if 'DIF' in df.columns:
        factors['macd_hist'] = df.get('MACD', df.get('DIF', 0) * 0)         # MACD柱
        factors['macd_dif']  = df.get('DIF', np.nan)                        # DIF值
    
    # ============ 波动因子 ============
    factors['volatility_5d']  = close.pct_change().rolling(5).std()         # 5日波动率
    factors['volatility_20d'] = close.pct_change().rolling(20).std()        # 20日波动率
    if 'BOLL_UP' in df.columns and 'BOLL_DOWN' in df.columns:
        factors['boll_width'] = (df['BOLL_UP'] - df['BOLL_DOWN']) / df['BOLL_MID']  # 带宽
    if 'ATR14' in df.columns:
        factors['atr_ratio'] = df['ATR14'] / close                            # ATR比率
    if 'K' in df.columns:
        factors['kdj_k'] = df['K']
        factors['kdj_d'] = df['D']
    
    return factors

In [ ]:
# 为所有测试股票计算因子
all_factors = {}
for code, ds in all_data.items():
    all_factors[code] = build_factors(ds, code)
    print(f'{code}: {all_factors[code].shape[1]} 个因子列')

# 合并为 MultiIndex DataFrame
factor_df = pd.concat(all_factors.values(), axis=0)
factor_df = factor_df.set_index('code', append=True).swaplevel(0, 1).sort_index()
print(f'\n合并后: {factor_df.shape[0]} 行 × {factor_df.shape[1]} 列')
print(f'因子列表: {[c for c in factor_df.columns if c not in ["close"]]}')

In [ ]:
# 快速可视化 — 随便挑一个股票看因子走势
code = TEST_CODES[0]
fig, axes = plt.subplots(3, 1, figsize=(16, 12))

# 价格 + 均线
ax = axes[0]
df = all_factors[code]
ax.plot(df.index.get_level_values(0), df['close'], label='Close', alpha=0.7)
ax.set_title(f'{code} 价格走势', fontsize=14)
ax.legend()
ax.grid(True, alpha=0.3)

# 动量因子
ax = axes[1]
ax.plot(df.index.get_level_values(0), df['momentum_20d'], label='20日动量', alpha=0.7)
ax.axhline(y=0, color='red', linestyle='--', alpha=0.5)
ax.set_title('动量因子: 20日收益率', fontsize=14)
ax.legend()
ax.grid(True, alpha=0.3)

# 波动因子
ax = axes[2]
ax.plot(df.index.get_level_values(0), df['volatility_20d'], label='20日波动率', alpha=0.7, color='orange')
ax.set_title('波动因子: 20日波动率', fontsize=14)
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## Part 3: IC 分析

计算每个因子对未来 N 日收益的预测能力（Spearman Rank IC）。

**IC > 0.02** 通常认为有预测力，**IR > 0.5** 认为稳定。

In [ ]:
# 3.1 构建未来收益序列
FORWARD_PERIODS = [1, 5, 20]  # 测试 1日/5日/20日 持有期

def build_forward_returns(close_panel, periods):
    """close_panel: code × date MultiIndex Series"""
    rets = {}
    for p in periods:
        rets[p] = close_panel.groupby(level=0).pct_change(p).shift(-p)
        rets[p].name = f'ret_{p}d'
    return rets

close_panel = factor_df['close']
forward_rets = build_forward_returns(close_panel, FORWARD_PERIODS)
print('未来收益已构建:', list(forward_rets.keys()))

In [ ]:
# 3.2 IC 计算引擎
def calc_ic_series(factor_series, forward_ret, min_stocks=3):
    """
    计算截面 IC（Spearman Rank Correlation）
    返回: pd.Series(index=date, values=IC)
    """
    # 对齐日期
    common_dates = factor_series.index.intersection(forward_ret.index)
    ic_values = {}
    
    for date in common_dates:
        f = factor_series.loc[date].dropna()
        r = forward_ret.loc[date].dropna()
        common_codes = f.index.intersection(r.index)
        
        if len(common_codes) >= min_stocks:
            ic, _ = spearmanr(f[common_codes], r[common_codes])
            if not np.isnan(ic):
                ic_values[date] = ic
    
    return pd.Series(ic_values, name='IC')


def ic_report(factor_name, ic_series):
    """打印 IC 报告"""
    mean_ic = ic_series.mean()
    std_ic  = ic_series.std()
    ir      = mean_ic / std_ic if std_ic > 0 else 0
    hit_rate = (ic_series > 0).mean()
    
    return {
        '因子': factor_name,
        'IC均值': round(mean_ic, 4),
        'IC标准差': round(std_ic, 4),
        'IR': round(ir, 4),
        'IC>0比例': f'{hit_rate:.1%}',
        '样本数': len(ic_series)
    }

In [ ]:
# 3.3 跑所有因子 × 所有持有期
factor_names = [c for c in factor_df.columns if c not in ['close', 'code']]

results = []
for factor_name in factor_names:
    factor_series = factor_df[factor_name]
    for period in FORWARD_PERIODS:
        try:
            ic = calc_ic_series(factor_series, forward_rets[period], min_stocks=2)
            if len(ic) > 0:
                report = ic_report(f'{factor_name}({period}d)', ic)
                results.append(report)
        except Exception as e:
            pass

# 按 IC 均值排序
results_df = pd.DataFrame(results).sort_values('IC均值', key=abs, ascending=False)
print(f'共 {len(results_df)} 组因子×持有期组合\n')
results_df

In [ ]:
# 3.4 可视化 — Top 因子 IC 累积曲线
top_factors = results_df.head(6)['因子'].tolist()
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, fname in enumerate(top_factors):
    ax = axes[i]
    parts = fname.rsplit('(', 1)
    fn = parts[0]
    period = int(parts[1].replace('d)', '')) if len(parts) > 1 else 5
    
    factor_series = factor_df[fn]
    ic = calc_ic_series(factor_series, forward_rets[period], min_stocks=2)
    cum_ic = ic.cumsum()
    ax.plot(cum_ic.index, cum_ic.values, linewidth=1.5)
    ax.axhline(y=0, color='red', linestyle='--', alpha=0.5)
    ax.set_title(f'{fname}\nIR={ic.mean()/ic.std():.3f}, Hit={(ic>0).mean():.1%}', fontsize=11)
    ax.grid(True, alpha=0.3)

plt.suptitle('Top 6 因子 × 持有期 — IC 累积曲线', fontsize=16, y=1.01)
plt.tight_layout()
plt.show()

---
## Part 4: 因子相关性检查

相似因子可以合并，减少过拟合风险。

In [ ]:
# 计算因子间相关性矩阵（取最新一天）
latest_date = factor_df.index.get_level_values(1).max()
latest_factors = factor_df.xs(latest_date, level=1).drop(columns=['close', 'code'], errors='ignore')

corr_matrix = latest_factors.corr()

fig, ax = plt.subplots(figsize=(12, 10))
im = ax.imshow(corr_matrix, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
ax.set_xticks(range(len(corr_matrix.columns)))
ax.set_xticklabels(corr_matrix.columns, rotation=45, ha='right', fontsize=9)
ax.set_yticks(range(len(corr_matrix.columns)))
ax.set_yticklabels(corr_matrix.columns, fontsize=9)
ax.set_title(f'因子相关性矩阵 ({str(latest_date)[:10]})', fontsize=14)
plt.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.show()

---
## Part 5: 策略回测

选择 IC 表现最好的因子，构建 CTA 策略跑回测。

In [ ]:
# 5.1 单标的均线趋势策略（基于 QAStrategyCtaBase）
from QUANTAXIS.QAStrategy.qactabase import QAStrategyCtaBase
from QUANTAXIS.QAUtil.QAParameter import MARKET_TYPE

class DualMAStrategy(QAStrategyCtaBase):
    """双均线趋势跟踪策略"""
    
    def user_init(self):
        self.ma_fast = 5
        self.ma_slow = 20
        self.pre_diff = 0
        self.trade_log = []
    
    def on_bar(self, bar):
        code = self.get_code()
        fast_val = bar.get(f'MA{self.ma_fast}', bar['close'])
        slow_val = bar.get(f'MA{self.ma_slow}', bar['close'])
        diff = fast_val - slow_val
        
        pos = self.get_positions(code)
        
        # 金叉买入
        if self.pre_diff <= 0 < diff:
            if pos.volume_long == 0:
                self.send_order('BUY', 'OPEN', bar['close'], 1000, code=code)
                self.trade_log.append({
                    'date': str(self.running_time)[:10],
                    'action': 'BUY',
                    'price': bar['close'],
                    'signal': 'golden_cross'
                })
        
        # 死叉卖出
        elif self.pre_diff >= 0 > diff:
            if pos.volume_long > 0:
                self.send_order('SELL', 'CLOSE', bar['close'], pos.volume_long, code=code)
                self.trade_log.append({
                    'date': str(self.running_time)[:10],
                    'action': 'SELL',
                    'price': bar['close'],
                    'signal': 'death_cross'
                })
        
        self.pre_diff = diff
    
    def on_dailyclose(self):
        pass

In [ ]:
# 5.2 回测 000001 平安银行
strategy = DualMAStrategy(
    code='000001',
    frequence='day',
    start='2020-01-01',
    end='2025-12-31',
    init_cash=1000000,
)

# 运行回测
strategy.run_backtest()

# 输出结果
qifi = strategy.acc.get_qifi()
print('=' * 50)
print(f'策略: 双均线趋势 (MA{strategy.ma_fast}/{strategy.ma_slow})')
print(f'标的: 000001 平安银行')
print(f'期间: 2020-01-01 ~ 2025-12-31')
print('=' * 50)
print(f'初始资金:   {strategy.init_cash:>15,.0f}')
print(f'期末权益:   {qifi["accounts"]["balance"]:>15,.2f}')
print(f'总收益率:   {(qifi["accounts"]["balance"]/strategy.init_cash - 1)*100:>14.2f}%')
print(f'平仓盈亏:   {qifi["accounts"]["close_profit"]:>15,.2f}')
print(f'浮动盈亏:   {qifi["accounts"]["float_profit"]:>15,.2f}')
print(f'手续费:     {qifi["accounts"]["commission"]:>15,.2f}')
print(f'交易次数:   {len(strategy.trade_log)//2} 次')
print(f'当前持仓:   {len(strategy.acc.positions)} 只')

In [ ]:
# 5.3 批量回测 — 多个标的上跑同一个策略
def run_single_backtest(code, strategy_class, **kwargs):
    """对单个标的运行回测，返回汇总指标"""
    try:
        s = strategy_class(code=code, frequence='day', **kwargs)
        s.run_backtest()
        qifi = s.acc.get_qifi()
        return {
            'code': code,
            'total_return': (qifi['accounts']['balance'] / kwargs['init_cash'] - 1) * 100,
            'close_profit': qifi['accounts']['close_profit'],
            'n_trades': len(s.trade_log) // 2 if hasattr(s, 'trade_log') else 0
        }
    except Exception as e:
        return {'code': code, 'total_return': None, 'error': str(e)}

# 跑多只
batch_results = []
for code in TEST_CODES[:4]:  # 先跑 4 只看效果
    r = run_single_backtest(code, DualMAStrategy,
                            start='2020-01-01', end='2025-12-31', init_cash=1000000)
    batch_results.append(r)
    status = f"{r['total_return']:+.2f}%" if r['total_return'] is not None else 'FAIL'
    print(f"{r['code']}: {status}")

batch_df = pd.DataFrame(batch_results)
print(f'\n平均收益率: {batch_df["total_return"].mean():.2f}%')

---
## Part 6: 从验证到实战的过渡

以上用 6 只股票验证了因子的 IC 有效性，下一步：

1. **扩大股票池** — 改为沪深 300 或全市场
2. **增加因子** — 在 `build_factors()` 中添加更多因子
3. **组合因子** — 选取 IC 显著的因子做等权合成
4. **全市场回测** — 每天选因子值最高的 N 只，等权买入

不需要修改任何 QA 源码即可完成。

In [ ]:
# 附录: 一键切换到全市场（沪深300）

# 获取沪深300成分股
from QUANTAXIS.QAFetch.QAQuery import QA_fetch_index_list

# 方式1: 从 stock_block 取沪深300
from QUANTAXIS.QAFetch.QAQuery import QA_fetch_stock_block
blocks = QA_fetch_stock_block()
print(f'板块列表: {blocks.columns.tolist() if hasattr(blocks, "columns") else type(blocks)}')

# 方式2: 手动筛选（上证60+深证60开头的部分代码）
all_stocks = QA_fetch_stock_list()
print(f'\n全市场共 {len(all_stocks)} 只股票')
print(f'\n将此处的 TEST_CODES 替换为全部代码即可切换全市场回测')
print(f'注意: 全市场回测耗时较长，建议先在小池子验证因子有效性')